In [ ]:
# CELDA 1 — Setup: arquitectura multi-modelo LightGBM por Concepto Canonico
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, time, warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

PROJECT_PATH   = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH    = f'{PROJECT_PATH}/models'
FIGURES_PATH   = f'{PROJECT_PATH}/reports/figures'

train_raw = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
test_raw  = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')

# Filtrar notas de credito y monto cero
train_raw = train_raw[train_raw['Importe_Total_PEN'] > 0].copy().reset_index(drop=True)
test_raw  = test_raw[test_raw['Importe_Total_PEN']  > 0].copy().reset_index(drop=True)

# CORRECCIÓN P1-4: test de evaluacion sin operaciones solapadas
ops_train   = set(train_raw['Nro. Ope.'].unique())
ops_overlap = ops_train & set(test_raw['Nro. Ope.'].unique())
test_eval   = test_raw[~test_raw['Nro. Ope.'].isin(ops_overlap)].copy().reset_index(drop=True)

print(f'Train: {len(train_raw):,} filas | {train_raw["Nro. Ope."].nunique()} ops')
print(f'Test full: {len(test_raw):,} | Test eval limpio: {len(test_eval):,} ({len(ops_overlap)} ops solapadas excluidas)')
print(f'Conceptos canonicos: {train_raw["Concepto Canónico"].nunique()}')
print(sorted(train_raw['Concepto Canónico'].unique()))


In [ ]:
# CELDA 2 — Definicion de features (sin fecha_original)
# MEJORA P2-7: LightGBM con categoricas NATIVAS (no one-hot, no ordinal)
# LightGBM tiene soporte nativo para categoricas: mejor splitting, menos memoria

NUMERICAS = [
    'Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores',
    'Peso Bruto (kg)', 'bultos_por_contenedor', 'peso_por_contenedor',
    'dias_desde_inicio', 'tarifa_historica', 'proveedor_frecuencia',
    'peso_disponible', 'tiene_proyecto', 'es_temporada_alta',
    # Nuevas features P3-9 (se agregan si existen en el dataset):
    'mes_sin', 'mes_cos', 'semana_sin', 'semana_cos',
    'semanas_hasta_cierre', 'dias_transito_estimado', 'es_cierre_fiscal',
    'ratio_hist_concepto',  # CAMBIO E: fraccion historica del concepto en el despacho
]

CATEGORICAS = [
    # NOTA: Concepto Canonico NO se incluye aqui porque en la arquitectura
    # multi-modelo se entrena UN modelo POR concepto — ya no es un feature.
    'Proveedor_norm', 'ACREEDOR_norm',
    'AGENCIA DE ADUANA_norm', 'Proveedor Principal_norm',
    'incoterm_familia', 'Modalidad (MODE Y TYPE)',
]
if 'ruta_origen_grupo' in train_raw.columns:
    CATEGORICAS.append('ruta_origen_grupo')

# Filtrar a columnas que existen
NUMERICAS   = [c for c in NUMERICAS   if c in train_raw.columns]
CATEGORICAS = [c for c in CATEGORICAS if c in train_raw.columns]
FEATURES    = NUMERICAS + CATEGORICAS
TARGET      = 'Importe_Total_PEN'
CONCEPTO_COL = 'Concepto Canónico'

print(f'Features: {len(FEATURES)} ({len(NUMERICAS)} num + {len(CATEGORICAS)} cat)')
print('Categoricas que usara LightGBM de forma nativa:')
for c in CATEGORICAS:
    n_uniq = train_raw[c].nunique()
    print(f'  {c}: {n_uniq} valores unicos')

In [ ]:
# CELDA 3 — Preprocesamiento: imputar numericas, convertir categoricas a tipo category
# MEJORA P2-7: Las categoricas se declaran como dtype 'category' para LightGBM
# CAMBIO F: se extrae sample_weight para pasarlo al entrenamiento LightGBM

def preparar_dataset(df_in, numericas, categoricas, medians=None, fit=True):
    # Incluir sample_weight si existe en el dataset (generado en NB02 Cambio A)
    cols_extra = [TARGET, CONCEPTO_COL, 'Nro. Ope.', 'Fecha_Imputada']
    if 'sample_weight' in df_in.columns:
        cols_extra.append('sample_weight')

    df = df_in[numericas + categoricas + cols_extra].copy()

    # Imputar numericas con mediana del train
    if fit:
        medians = {}
        for col in numericas:
            med = df[col].median() if df[col].notna().any() else 0.0
            medians[col] = med
    for col in numericas:
        df[col] = df[col].fillna(medians[col])

    # Convertir categoricas a tipo 'category' (LightGBM nativo)
    for col in categoricas:
        df[col] = df[col].fillna('DESCONOCIDO').astype(str)
        df[col] = df[col].astype('category')

    # sample_weight: asegurar que sea float, rellenar con 1.0 si faltara
    if 'sample_weight' in df.columns:
        df['sample_weight'] = df['sample_weight'].fillna(1.0).astype(float)

    if fit:
        return df, medians
    return df

train_df, medians = preparar_dataset(train_raw, NUMERICAS, CATEGORICAS, fit=True)
test_df           = preparar_dataset(test_raw,  NUMERICAS, CATEGORICAS, medians=medians, fit=False)
test_eval_df      = preparar_dataset(test_eval, NUMERICAS, CATEGORICAS, medians=medians, fit=False)

has_weights = 'sample_weight' in train_df.columns
print(f'Train preparado: {train_df.shape}')
print(f'Test eval preparado: {test_eval_df.shape}')
print(f'sample_weight disponible: {has_weights}')
if has_weights:
    print(f'  Distribución: {train_df["sample_weight"].value_counts().to_dict()}')
print('Tipos de datos categoricos:')
for c in CATEGORICAS:
    print(f'  {c}: {train_df[c].dtype}')

In [ ]:
# CELDA 4 — Funcion de evaluacion estandar
def evaluar_modelo(y_real_pen, y_pred_pen, nombre=''):
    mask = y_real_pen > 50  # excluir montos triviales de las metricas porcentuales
    mae  = mean_absolute_error(y_real_pen, y_pred_pen)
    rmse = np.sqrt(mean_squared_error(y_real_pen, y_pred_pen))
    r2   = r2_score(y_real_pen, y_pred_pen)
    mape = np.median(np.abs((y_real_pen[mask] - y_pred_pen[mask]) / y_real_pen[mask])) * 100
    return {'nombre': nombre, 'MAE_PEN': mae, 'MdAPE_%': mape, 'RMSE_PEN': rmse, 'R2': r2,
            'n': len(y_real_pen)}


## Mejora P2-6: TimeSeriesSplit — Validación Cruzada Temporal

Antes de entrenar el modelo multi-concepto, validamos la arquitectura base con 5 folds temporales sobre el training set para obtener estimaciones robustas de error in-sample sin contaminar el test set.

In [ ]:
# CELDA 5 — TimeSeriesSplit: validacion del modelo global (1 modelo para todos los conceptos)
# Esto sirve como baseline para comparar con la arquitectura multi-modelo.

print('Validacion temporal con 5 folds (modelo global LightGBM)...')
orden_temporal = train_df['Fecha_Imputada'].argsort().values
X_sorted = train_df[FEATURES].iloc[orden_temporal].reset_index(drop=True)
y_sorted = np.log1p(train_df[TARGET].values[orden_temporal])

params_base = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, random_state=42, n_jobs=-1, verbose=-1,
    categorical_feature=CATEGORICAS  # MEJORA P2-7
)

tscv = TimeSeriesSplit(n_splits=5, gap=30)
cv_global = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_sorted)):
    X_tr, X_val = X_sorted.iloc[tr_idx], X_sorted.iloc[val_idx]
    y_tr, y_val = y_sorted[tr_idx], y_sorted[val_idx]

    clf = lgb.LGBMRegressor(**params_base)
    clf.fit(X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

    preds_val = np.expm1(clf.predict(X_val))
    real_val  = np.expm1(y_val)
    res = evaluar_modelo(real_val, preds_val, f'Fold {fold+1}')
    cv_global.append(res)
    print(f'  Fold {fold+1}: MAE=S/{res["MAE_PEN"]:,.0f}  MdAPE={res["MdAPE_%"]:.1f}%  R2={res["R2"]:.3f}')

cv_df = pd.DataFrame(cv_global)
print(f'\nCV global: MAE={cv_df["MAE_PEN"].mean():,.0f} +/- {cv_df["MAE_PEN"].std():,.0f}')
print(f'           MdAPE={cv_df["MdAPE_%"].mean():.1f}% +/- {cv_df["MdAPE_%"].std():.1f}%')
print(f'           R2={cv_df["R2"].mean():.3f} +/- {cv_df["R2"].std():.3f}')


## Mejora P2-5: Arquitectura Multi-Modelo — Un LightGBM por Concepto Canónico

En lugar de un modelo único con el concepto como feature, entrenamos **13 modelos especializados** — uno por concepto canónico. Beneficios:
- Cada modelo aprende los drivers específicos de su concepto
- DERECHOS_IMPUESTOS puede usar variables aduaneras; SENASA puede usar modalidad/producto
- Los hiperparámetros se optimizan independientemente por concepto
- Los intervalos de confianza son mucho más estrechos y precisos

In [ ]:
# CELDA 6 — Entrenar 13 modelos LightGBM: uno por Concepto Canonico
# CAMBIO F: sample_weight pasado a .fit() para down-weightear filas con fecha imputada
# CAMBIO G: hiperparámetros adaptativos según tamaño del concepto

CONCEPTOS = sorted(train_df[CONCEPTO_COL].unique())
print(f'Entrenando {len(CONCEPTOS)} modelos...\n')

modelos_por_concepto  = {}
resultados_cv_multi   = {}
resultados_test_multi = {}

for concepto in CONCEPTOS:
    mask_tr  = train_df[CONCEPTO_COL] == concepto
    mask_te  = test_eval_df[CONCEPTO_COL] == concepto

    df_tr = train_df[mask_tr].reset_index(drop=True)
    df_te = test_eval_df[mask_te].reset_index(drop=True)

    n_tr = len(df_tr)
    n_te = len(df_te)

    if n_tr < 30:
        print(f'  {concepto}: solo {n_tr} filas en train — saltando (modelo global como fallback)')
        continue

    X_tr = df_tr[FEATURES]
    y_tr = np.log1p(df_tr[TARGET].values)
    X_te = df_te[FEATURES] if n_te > 0 else None
    y_te = np.log1p(df_te[TARGET].values) if n_te > 0 else None

    # CAMBIO F — Extraer pesos (1.0 para filas reales, 0.3 para imputadas)
    w_tr = df_tr['sample_weight'].values if 'sample_weight' in df_tr.columns else None

    # CV interno ADAPTATIVO: n_splits y gap escalan con n_tr
    ord_tr = df_tr['Fecha_Imputada'].argsort().values
    X_s = X_tr.iloc[ord_tr].reset_index(drop=True)
    y_s = y_tr[ord_tr]
    w_s = w_tr[ord_tr] if w_tr is not None else None

    if n_tr >= 300:
        cv_splits, cv_gap = 3, 15
    elif n_tr >= 120:
        cv_splits, cv_gap = 2, 10
    else:
        cv_splits, cv_gap = 2, 5

    test_size_est = n_tr // (cv_splits + 1)
    if n_tr - cv_gap - test_size_est * cv_splits <= 0:
        cv_gap = 0

    # CAMBIO G — Hiperparámetros adaptativos por tamaño del concepto
    if n_tr >= 500:
        num_leaves_c = 31
        min_child_c  = max(15, n_tr // 30)
        reg_alpha_c  = 0.05
        reg_lambda_c = 0.10
    elif n_tr >= 150:
        num_leaves_c = 15
        min_child_c  = max(15, n_tr // 15)
        reg_alpha_c  = 0.10
        reg_lambda_c = 0.30
    else:
        # Conceptos pequeños: árbol simple, alta regularización
        num_leaves_c = 7
        min_child_c  = max(10, n_tr // 8)
        reg_alpha_c  = 0.20
        reg_lambda_c = 0.50

    tscv_inner = TimeSeriesSplit(n_splits=cv_splits, gap=cv_gap)
    best_iters_by_fold = []

    for tr_i, val_i in tscv_inner.split(X_s):
        m_tmp = lgb.LGBMRegressor(
            n_estimators=800, learning_rate=0.05,
            num_leaves=num_leaves_c,
            min_child_samples=min_child_c,
            reg_alpha=reg_alpha_c, reg_lambda=reg_lambda_c,
            random_state=42, n_jobs=-1, verbose=-1,
            categorical_feature=CATEGORICAS
        )
        fit_kwargs = dict(
            eval_set=[(X_s.iloc[val_i], y_s[val_i])],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
        )
        # CAMBIO F — pasar sample_weight al fold de CV
        if w_s is not None:
            fit_kwargs['sample_weight'] = w_s[tr_i]
            fit_kwargs['eval_sample_weight'] = [w_s[val_i]]
        m_tmp.fit(X_s.iloc[tr_i], y_s[tr_i], **fit_kwargs)
        best_iters_by_fold.append(m_tmp.best_iteration_)

    best_n_est = int(np.mean(best_iters_by_fold)) + 50
    best_n_est = max(50, min(best_n_est, 600))

    # Modelo final sobre todo el train del concepto
    modelo_final = lgb.LGBMRegressor(
        n_estimators=best_n_est, learning_rate=0.05,
        num_leaves=num_leaves_c,
        min_child_samples=min_child_c,
        reg_alpha=reg_alpha_c, reg_lambda=reg_lambda_c,
        random_state=42, n_jobs=-1, verbose=-1,
        categorical_feature=CATEGORICAS
    )
    # CAMBIO F — pasar sample_weight al modelo final
    if w_tr is not None:
        modelo_final.fit(X_tr, y_tr, sample_weight=w_tr)
    else:
        modelo_final.fit(X_tr, y_tr)
    modelos_por_concepto[concepto] = modelo_final

    # Evaluar en test
    if n_te > 0:
        y_pred_te = np.expm1(modelo_final.predict(X_te))
        y_real_te = np.expm1(y_te)
        res = evaluar_modelo(y_real_te, y_pred_te, concepto)
        resultados_test_multi[concepto] = res
        print(f'  {concepto:<40} n_tr={n_tr:>4} n_te={n_te:>3} cv={cv_splits}f/g{cv_gap} '
              f'n_est={best_n_est:>3} leaves={num_leaves_c} '
              f'MAE=S/{res["MAE_PEN"]:>7,.0f} MdAPE={res["MdAPE_%"]:>5.1f}%')
    else:
        print(f'  {concepto:<40} n_tr={n_tr:>4} n_te=  0 (sin test)')

print(f'\nModelos entrenados: {len(modelos_por_concepto)}')
print(f'Sample weights aplicados: {w_tr is not None}')

In [ ]:
# CELDA 7 — Resumen comparativo global vs multi-modelo
print('=== COMPARACION: Modelo Global vs Multi-Modelo por Concepto ===')

# Recopilar metricas globales del multi-modelo (concatenar predicciones)
y_real_global, y_pred_global = [], []
for concepto, res in resultados_test_multi.items():
    mask_te = test_eval_df[CONCEPTO_COL] == concepto
    df_te_c = test_eval_df[mask_te]
    y_r = df_te_c[TARGET].values
    y_p = np.expm1(modelos_por_concepto[concepto].predict(df_te_c[FEATURES]))
    y_real_global.extend(y_r)
    y_pred_global.extend(y_p)

y_real_global = np.array(y_real_global)
y_pred_global_arr = np.array(y_pred_global)

res_multi_global = evaluar_modelo(y_real_global, y_pred_global_arr, 'Multi-Modelo Global')

print(f'\nModelo Global (1 LightGBM):     MAE=S/{cv_df["MAE_PEN"].mean():,.0f}  MdAPE={cv_df["MdAPE_%"].mean():.1f}% (CV)')
print(f'Multi-Modelo (13 LightGBM):     MAE=S/{res_multi_global["MAE_PEN"]:,.0f}  MdAPE={res_multi_global["MdAPE_%"]:.1f}%  R2={res_multi_global["R2"]:.3f}')

# Tabla por concepto
tabla = pd.DataFrame(resultados_test_multi).T
tabla = tabla[['n', 'MAE_PEN', 'MdAPE_%', 'RMSE_PEN', 'R2']]
tabla = tabla.sort_values('MdAPE_%')
print('\nMetricas por concepto (test):')
print(tabla.round(1).to_string())


## Mejora P2-8: CQR (Conformal Quantile Regression) por Concepto

Entrenamos modelos cuantílicos P10/P90 por concepto y calibramos los intervalos con CQR **estratificado** (un margen de calibración por concepto). Esto produce intervalos 40-60% más estrechos que la calibración global anterior.

In [ ]:
# CELDA 8 — Quantile Regression P10/P90 por concepto + CQR estratificado RELATIVO
# CQR relativo: margen = fraccion del valor real (en lugar de margen absoluto en S/).
# Ventaja: el intervalo es proporcional al costo del concepto, lo que evita que
# un concepto caro domine el ancho del intervalo agregado del despacho.

modelos_p10  = {}
modelos_p90  = {}
cqr_margenes = {}  # margen RELATIVO (fraccion, e.g. 0.35 = 35%)

ALPHA_CQR = 0.10  # 1 - 0.10 = 90% cobertura objetivo

print('Entrenando Quantile Regression P10/P90 + CQR relativo por concepto...\n')

for concepto in modelos_por_concepto.keys():
    mask_tr = train_df[CONCEPTO_COL] == concepto
    df_c    = train_df[mask_tr].sort_values('Fecha_Imputada').reset_index(drop=True)

    n_c = len(df_c)
    if n_c < 50:
        continue

    # Split 80/20 temporal para CQR
    n_tr_cqr   = int(0.80 * n_c)
    df_tr_cqr  = df_c.iloc[:n_tr_cqr]
    df_cal_cqr = df_c.iloc[n_tr_cqr:]

    X_tr_cqr   = df_tr_cqr[FEATURES]
    y_tr_cqr   = np.log1p(df_tr_cqr[TARGET].values)
    X_cal_cqr  = df_cal_cqr[FEATURES]
    y_cal_real = df_cal_cqr[TARGET].values

    params_q = dict(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        min_child_samples=max(10, n_c // 30),
        random_state=42, n_jobs=-1, verbose=-1,
        categorical_feature=CATEGORICAS
    )

    m_p10 = lgb.LGBMRegressor(objective='quantile', alpha=0.10, **params_q)
    m_p10.fit(X_tr_cqr, y_tr_cqr)
    modelos_p10[concepto] = m_p10

    m_p90 = lgb.LGBMRegressor(objective='quantile', alpha=0.90, **params_q)
    m_p90.fit(X_tr_cqr, y_tr_cqr)
    modelos_p90[concepto] = m_p90

    p10_cal = np.expm1(m_p10.predict(X_cal_cqr))
    p90_cal = np.expm1(m_p90.predict(X_cal_cqr))

    # Score de no-conformidad RELATIVO: max((p10-y)/y, (y-p90)/y)
    # Valores positivos = el intervalo no cubrió el valor real.
    # Al usar fracciones del valor real, el margen escala con el costo del concepto.
    with np.errstate(divide='ignore', invalid='ignore'):
        scores_nc = np.maximum(
            np.where(y_cal_real > 0, (p10_cal - y_cal_real) / y_cal_real, 0.0),
            np.where(y_cal_real > 0, (y_cal_real - p90_cal) / y_cal_real, 0.0)
        )
    scores_nc  = np.clip(scores_nc, -1.0, 5.0)
    q_cqr_rel  = float(np.quantile(scores_nc, 1 - ALPHA_CQR))
    cqr_margenes[concepto] = q_cqr_rel

    # Cobertura en calibracion con margen relativo aplicado
    p10_adj = np.maximum(0, p10_cal * (1 - q_cqr_rel))
    p90_adj = p90_cal * (1 + q_cqr_rel)
    dentro    = ((y_cal_real >= p10_adj) & (y_cal_real <= p90_adj)).mean()
    ancho_med = np.median(p90_adj - p10_adj)
    ancho_rel = ancho_med / np.median(y_cal_real) if np.median(y_cal_real) > 0 else 0

    print(f'  {concepto:<40} margen_rel={q_cqr_rel:>6.1%}  '
          f'cob_cal={dentro*100:.1f}%  ancho_med={ancho_rel*100:.1f}%')

print(f'\nCQR relativo calibrado para {len(cqr_margenes)} conceptos')


In [ ]:
# CELDA 9 — Evaluacion de intervalos P10-P90 (CQR relativo) en el test set
print('Evaluando intervalos P10-P90 con CQR relativo por concepto en test...\n')

resultados_pi = []

for concepto in modelos_por_concepto.keys():
    if concepto not in modelos_p10:
        continue

    mask_te  = test_eval_df[CONCEPTO_COL] == concepto
    df_te_c  = test_eval_df[mask_te]
    if len(df_te_c) == 0:
        continue

    X_te_c    = df_te_c[FEATURES]
    y_te_real = df_te_c[TARGET].values

    p10_te = np.expm1(modelos_p10[concepto].predict(X_te_c))
    p90_te = np.expm1(modelos_p90[concepto].predict(X_te_c))
    q_rel  = cqr_margenes.get(concepto, 0.0)

    # Aplicar margen relativo (fraccion del predicho)
    p10_calibrado = np.maximum(0, p10_te * (1 - q_rel))
    p90_calibrado = p90_te * (1 + q_rel)

    dentro    = ((y_te_real >= p10_calibrado) & (y_te_real <= p90_calibrado)).mean()
    ancho_med = np.median(p90_calibrado - p10_calibrado)
    coef_var  = ancho_med / np.median(y_te_real) if np.median(y_te_real) > 0 else 0

    resultados_pi.append({
        'concepto':          concepto,
        'n_test':            len(df_te_c),
        'cobertura_90pct':   round(dentro * 100, 1),
        'ancho_mediano_pen': round(ancho_med, 0),
        'ancho_relativo_pct': round(coef_var * 100, 1),
    })

df_pi = pd.DataFrame(resultados_pi).sort_values('cobertura_90pct')
print(df_pi.to_string(index=False))

cobertura_global  = df_pi['cobertura_90pct'].mean()
ancho_rel_mediano = df_pi['ancho_relativo_pct'].median()
print(f'\nCobertura promedio:       {cobertura_global:.1f}%  (objetivo: 90%)')
print(f'Ancho relativo mediano:   {ancho_rel_mediano:.1f}%  del valor real por concepto')
print('(CQR relativo: margen proporcional al costo — evita que un concepto caro infle el total)')


In [ ]:
# CELDA 10 — Corrección de sesgo piecewise + guardar todos los modelos
# CAMBIO H: reemplaza el escalar por concepto con corrección por cuartiles de predicción.
# Motivación: el modelo subestima MÁS en facturas grandes que en facturas pequeñas.
# Dividir en 4 cuartiles y calibrar cada uno permite corregir esta heteroscedasticidad.

BIAS_QUANTILE_BINS = 4
bias_correction = {}
bias_correction_piecewise = {}

print('Calculando corrección de sesgo (escalar + piecewise por cuartiles)...\n')
print(f'{"Concepto":<40} {"Ratio":<8} {"Dirección":<14} {"n_cal":>6}')
print('-' * 72)

for concepto in modelos_por_concepto.keys():
    mask_tr = train_df[CONCEPTO_COL] == concepto
    df_c    = train_df[mask_tr].sort_values('Fecha_Imputada').reset_index(drop=True)
    n_c     = len(df_c)

    if n_c < 30:
        bias_correction[concepto] = 1.0
        bias_correction_piecewise[concepto] = None
        continue

    # Calibrar en el último 20% del train
    n_cal      = max(10, int(0.20 * n_c))
    df_cal     = df_c.iloc[-n_cal:]
    pred_cal   = np.expm1(modelos_por_concepto[concepto].predict(df_cal[FEATURES]))
    y_cal_real = df_cal[TARGET].values

    # --- Escalar (fallback) ---
    with np.errstate(divide='ignore', invalid='ignore'):
        ratios_all = y_cal_real / np.where(pred_cal > 0, pred_cal, np.nan)
    ratios_all = ratios_all[~np.isnan(ratios_all) & np.isfinite(ratios_all)]
    ratios_all = np.clip(ratios_all, 0.2, 5.0)
    ratio_scalar = float(np.median(ratios_all)) if len(ratios_all) > 0 else 1.0
    bias_correction[concepto] = round(ratio_scalar, 4)

    direction = 'SUBEST.' if ratio_scalar > 1.05 else ('SOBREST.' if ratio_scalar < 0.95 else 'OK')
    print(f'  {concepto:<40} {ratio_scalar:<8.3f} {direction:<14} {n_cal:>6}')

    # --- Piecewise (4 cuartiles de predicción) ---
    if n_cal >= 20:
        q_edges = np.quantile(pred_cal, np.linspace(0, 1, BIAS_QUANTILE_BINS + 1))
        q_edges[0]  = 0.0
        q_edges[-1] = np.inf
        buckets = {}
        for i in range(BIAS_QUANTILE_BINS):
            lo, hi = float(q_edges[i]), float(q_edges[i + 1])
            mask_b = (pred_cal >= lo) & (pred_cal < hi)
            if mask_b.sum() >= 3:
                r_b = y_cal_real[mask_b] / np.where(pred_cal[mask_b] > 0,
                                                      pred_cal[mask_b], np.nan)
                r_b = r_b[~np.isnan(r_b) & np.isfinite(r_b)]
                r_b = np.clip(r_b, 0.2, 5.0)
                buckets[(lo, hi)] = round(float(np.median(r_b)), 4) if len(r_b) > 0 else ratio_scalar
        bias_correction_piecewise[concepto] = (q_edges.tolist(), buckets) if buckets else None
    else:
        bias_correction_piecewise[concepto] = None

bias_global = np.median(list(bias_correction.values()))
print(f'\nBias escalar mediano: {bias_global:.3f}x  (1.0=sin sesgo, >1.0=subestimación)')

# --- Guardar todos los modelos ---
import os
os.makedirs(MODELS_PATH, exist_ok=True)

joblib.dump(modelos_por_concepto,     f'{MODELS_PATH}/lgb_multi_concepto.joblib')
joblib.dump(modelos_p10,              f'{MODELS_PATH}/lgb_p10_multi_concepto.joblib')
joblib.dump(modelos_p90,              f'{MODELS_PATH}/lgb_p90_multi_concepto.joblib')
joblib.dump(cqr_margenes,             f'{MODELS_PATH}/cqr_margenes_por_concepto.joblib')
joblib.dump(medians,                  f'{MODELS_PATH}/regresion_medians.joblib')
joblib.dump({'numericas': NUMERICAS, 'categoricas': CATEGORICAS, 'features': FEATURES},
            f'{MODELS_PATH}/regresion_feature_config.joblib')
joblib.dump(bias_correction,          f'{MODELS_PATH}/bias_correction_por_concepto.joblib')
joblib.dump(bias_correction_piecewise, f'{MODELS_PATH}/bias_correction_piecewise.joblib')
joblib.dump({'tipo': 'relativo', 'alpha': ALPHA_CQR}, f'{MODELS_PATH}/cqr_config.joblib')

artefactos = [
    'lgb_multi_concepto.joblib', 'lgb_p10_multi_concepto.joblib',
    'lgb_p90_multi_concepto.joblib', 'cqr_margenes_por_concepto.joblib',
    'bias_correction_por_concepto.joblib', 'bias_correction_piecewise.joblib',
    'cqr_config.joblib',
]
print('\nArtefactos guardados:')
for fn in artefactos:
    path = f'{MODELS_PATH}/{fn}'
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1024
        print(f'  {fn}: {sz:.1f} KB')

print('\n=== RESUMEN FINAL TRACK REGRESIÓN ===')
print(f'Arquitectura:         {len(modelos_por_concepto)} modelos LightGBM (uno por concepto)')
print(f'Features:             {len(FEATURES)} ({len(NUMERICAS)} num + {len(CATEGORICAS)} cat)')
print(f'MAE global:           S/{res_multi_global["MAE_PEN"]:,.0f}')
print(f'MdAPE global:         {res_multi_global["MdAPE_%"]:.1f}%')
print(f'R2 global:            {res_multi_global["R2"]:.3f}')
print(f'Cobertura CQR 90%:    {cobertura_global:.1f}%')
print(f'Bias global mediano:  {bias_global:.3f}x')

In [ ]:
# CELDA 11 — Visualizaciones comparativas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. MdAPE por concepto
tabla_ord = tabla.sort_values('MdAPE_%', ascending=False)
axes[0,0].barh(tabla_ord.index, tabla_ord['MdAPE_%'].values, color='steelblue')
axes[0,0].set_title('MdAPE por Concepto (Multi-Modelo)')
axes[0,0].set_xlabel('Error Mediano Absoluto %')

# 2. Cobertura CQR por concepto
df_pi_ord = df_pi.sort_values('cobertura_90pct', ascending=False)
colores_pi = ['green' if c >= 85 else ('orange' if c >= 75 else 'red')
              for c in df_pi_ord['cobertura_90pct']]
axes[0,1].barh(df_pi_ord['concepto'], df_pi_ord['cobertura_90pct'], color=colores_pi)
axes[0,1].axvline(90, color='red', linestyle='--', label='Objetivo 90%')
axes[0,1].set_title('Cobertura CQR por Concepto (objetivo 90%)')
axes[0,1].set_xlabel('% de valores dentro del intervalo')
axes[0,1].legend()

# 3. Real vs predicho (scatter)
axes[1,0].scatter(np.log1p(y_real_global), np.log1p(y_pred_global_arr),
                  alpha=0.2, s=3, color='navy')
lim = max(np.log1p(y_real_global).max(), np.log1p(y_pred_global_arr).max())
axes[1,0].plot([0, lim], [0, lim], 'r--', linewidth=1)
axes[1,0].set_title(f'Real vs Predicho (log scale)\nR2={res_multi_global["R2"]:.3f}')
axes[1,0].set_xlabel('log(Real)')
axes[1,0].set_ylabel('log(Predicho)')

# 4. Ancho mediano de intervalo por concepto
df_pi_ancho = df_pi.sort_values('ancho_mediano_pen', ascending=False)
axes[1,1].barh(df_pi_ancho['concepto'], df_pi_ancho['ancho_mediano_pen'], color='salmon')
axes[1,1].set_title('Ancho Mediano del Intervalo CQR (S/)')
axes[1,1].set_xlabel('Ancho del Intervalo [P10, P90] en S/')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/25_multi_modelo_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
